# Run underfit in Colab

**by [@dadabots](https://dadabots.com)** · last updated: 2026-05-25 07:00 UTC
[github.com/dada-bots/underfit](https://github.com/dada-bots/underfit)

LoRA finetuning for Stable Audio 3, in a Colab notebook.

> ⚠️ **Before running any cells:** set the runtime to GPU.
> **Runtime → Change runtime type → T4 GPU** (or **L4** / **H100** if you have Colab Pro / Pro+).

The dashboard runs *inside* the Colab VM on `localhost:8787` and is exposed via the Colab port-proxy (private to your Google account) or via ngrok (public link).

Run each cell with `Shift+Enter`.


## 1. Install underfit

In [ ]:
# ── PRE-FLIGHT: confirm GPU runtime is attached ─────────────────────────
import subprocess
try:
    r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                       capture_output=True, text=True, timeout=5)
    if r.returncode != 0 or not r.stdout.strip():
        raise FileNotFoundError
    print(f"✓ GPU detected: {r.stdout.strip().splitlines()[0]}")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("\n" + "═" * 70)
    print("  ⚠️  NO GPU DETECTED — underfit needs a GPU runtime  ⚠️")
    print("═" * 70)
    print("\n  Fix it now:")
    print("    1. Runtime → Change runtime type")
    print("    2. Hardware accelerator → T4 GPU  (or L4 / H100 if you have Pro)")
    print("    3. Click Save (your VM will reconnect)")
    print("    4. Re-run this cell")
    print("\n" + "═" * 70 + "\n")
    # Banner is already printed; SystemExit() with no message avoids tripping
    # IPython's traceback formatter on Python 3.12.
    raise SystemExit()

# ── Clone underfit + install deps (uv + venv, ~5 GB of wheels, ~1 min) ──
import os
if not os.path.isdir("/content/underfit"):
    !git clone https://github.com/dada-bots/underfit /content/underfit
%cd /content/underfit
!./install.sh --no-setup


## 2. Mount Drive + HF auth + state dir

Splits storage between two locations to balance persistence and speed:

| Lives on | Why |
|---|---|
| **Drive** (`UNDERFIT_STATE_DIR`) | runs.json, datasets, audio, gradio logs — small files, must persist across session resets |
| **Drive** (`HF_HOME`) | HuggingFace cache (~15 GB of model weights) — avoids re-downloading from HF each session |
| **Local SSD** (`UNDERFIT_MODELS_DIR=/content/models`) | Materialized model files used by training. Reads at ~500 MB/s vs Drive's ~30 MB/s. Wiped on session reset, but re-staged in ~5 min by the wizard. |


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Persistent user state (runs, datasets, audio) — lives on Drive so it
# survives Colab session resets.
os.environ['UNDERFIT_STATE_DIR'] = '/content/drive/MyDrive/underfit-state'

# Model checkpoints — stay on local SSD for fast reads (~500 MB/s vs Drive's
# ~30 MB/s through FUSE). Wiped on session reset, but re-staged in ~5 min by
# the wizard's snapshot_download from the HF cache (which IS on Drive).
os.environ['UNDERFIT_MODELS_DIR'] = '/content/models'

# HuggingFace cache — on Drive so models don't re-download from HF each session.
# The wizard copies from this cache to UNDERFIT_MODELS_DIR (local SSD).
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf-cache'

# Authenticate with HuggingFace — needs access to gated stabilityai/stable-audio-3-* repos.
# Paste your token from https://huggingface.co/settings/tokens when prompted.
print("⚠️  huggingface_hub may first ask 'A new version is available — do you want to update?'")
print("    You probably don't need to. Type 'n' and press Enter.")
print()
!hf auth login


## 3. Pick a backend + download model packs

This clones the Stable Audio 3 backend into `/content/stable-audio-3`, installs it into the venv, and downloads the model packs you ask for. Replace `sa3-medium` with `sa3-sm-music`, `sa3-sm-sfx`, or any comma-separated combination.

> ⚠️ **First time only:** open each model page below and click **"Agree and access repository"** — both the base AND the ARC variant. Without this, the download in this cell will fail with `401 Unauthorized`. (Approval is instant once you accept the license.)
>
> - **sa3-medium** — [base](https://huggingface.co/stabilityai/stable-audio-3-medium-base) · [ARC](https://huggingface.co/stabilityai/stable-audio-3-medium)
> - **sa3-sm-music** — [base](https://huggingface.co/stabilityai/stable-audio-3-small-music-base) · [ARC](https://huggingface.co/stabilityai/stable-audio-3-small-music)
> - **sa3-sm-sfx** — [base](https://huggingface.co/stabilityai/stable-audio-3-small-sfx-base) · [ARC](https://huggingface.co/stabilityai/stable-audio-3-small-sfx)


In [ ]:
# Clone the SA3 backend if not already present
import os
if not os.path.isdir("/content/stable-audio-3"):
    !git clone https://github.com/Stability-AI/stable-audio-3.git /content/stable-audio-3

# Run the setup wizard non-interactively
!uv run python -m underfit.cli.setup \
    --backend sa3 \
    --backend-path /content/stable-audio-3 \
    --models sa3-medium


## 4. Launch the dashboard

Two ways to reach the dashboard once it's running:

- **Private (default):** Colab's built-in port-proxy. Only your Google account can open the URL. No third-party service.
- **Shareable:** ngrok tunnel — public URL anyone with the link can use. Free tier requires a token from [ngrok.com](https://ngrok.com/signup).

This cell **completes** after the dashboard starts (the subprocess keeps running in the background). Step 5 is a separate cell that polls the dashboard for status updates.


In [ ]:
import subprocess
from underfit.monitor import launch_dashboard_subprocess, dashboard_button, restart_dashboard_button

# ┌─ Toggle to share the dashboard with collaborators ──────────────────┐
SHARE_PUBLICLY = False
NGROK_AUTHTOKEN = ""   # only needed if SHARE_PUBLICLY = True
# └─────────────────────────────────────────────────────────────────────┘

# launch_dashboard_subprocess() kills any prior process bound to :8787,
# launches dashboard/server.py with start_new_session=True (so cell
# interrupts don't propagate), and blocks until the "Dashboard running on"
# ready-marker. Returns the Popen handle.
proc = launch_dashboard_subprocess(port=8787)

# Render the big "Open Dashboard" button
if SHARE_PUBLICLY and NGROK_AUTHTOKEN:
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyngrok"])
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_AUTHTOKEN
    ngrok_url = ngrok.connect(8787, 'http').public_url
    dashboard_button(url=ngrok_url, label="🌐 Open Public Dashboard (ngrok)")
else:
    dashboard_button(port=8787)

# Secondary action below the big Open button.
restart_dashboard_button(port=8787)

print("\n(Dashboard is running in the background. Move on to Step 5 to monitor it.)")


## 5. Monitor the dashboard (optional)

Polls the dashboard's API every 10 seconds and shows: runs by status, dataset count, active gradio instances. Doubles as a keep-alive so Colab doesn't drop the session.

If the dashboard subprocess crashes (or gets OOM-killed mid-training), the monitor **auto-restarts it** after ~30 s of consecutive failed polls. You won't lose your runs — `runs.json` lives on Drive, the dashboard re-reads it on launch.

This cell is **decoupled** from the dashboard subprocess — stop the monitor anytime (⏹ / `Ctrl+M I`) without killing the dashboard. Re-run this cell to restart the monitor.


In [ ]:
from underfit.monitor import launch_dashboard_subprocess, dashboard_button, start_monitor

def _restart_dashboard():
    """Called by the monitor after ~30s of failed polls. Only restarts if the
    subprocess is actually dead — otherwise the dashboard is alive but slow
    under load (training hammering GPU/CPU), and killing it makes things worse."""
    global proc
    if proc.poll() is None:
        print("ℹ  Dashboard slow but PID is still alive — skipping restart.", flush=True)
        return
    print(f"⚠️  Dashboard subprocess died (returncode={proc.returncode}). Restarting…", flush=True)
    proc = launch_dashboard_subprocess(port=8787, quiet=True)
    print("✓ Dashboard restarted (same URL — button above still works).\n", flush=True)

# Refreshes every 10 s. Stop with the ⏹ button; the dashboard keeps running.
# Auto-restarts the dashboard only if its process actually crashed.
dashboard_button(port=8787)
start_monitor(interval=10, on_unreachable=_restart_dashboard)


## 6. Create your first dataset

In the dashboard tab, click **"+ Dataset"** to encode a folder of audio into pre-encoded latents.

### Get your audio onto Google Drive

1. Open [drive.google.com](https://drive.google.com) in another tab.
2. Drag-and-drop a folder of audio (e.g. `my-songs/`) into "My Drive".
3. Wait for the green check on every file.
4. In the dashboard's **Create Dataset** form, paste the path:
   ```
   /content/drive/MyDrive/my-songs
   ```
   (replace `my-songs` with your folder's actual name)

Verify the path from a Colab cell if you want:
```python
!ls /content/drive/MyDrive/my-songs | head
```

### Tips on what to feed it

- **10+ minutes** of audio is the floor; **30+ min** is better. Quality > quantity.
- **WAV / FLAC / MP3 / OGG / OPUS / M4A / AIFF** all work.
- **One coherent style** per dataset (one artist, one genre, one SFX category).
- The dashboard lets you tick/untick individual files after the scan — you don't have to pre-curate.

### What happens when you submit

The dashboard spawns a pre-encoding subprocess on the GPU. Each audio file becomes a `.npy` (encoded latent) + `.json` (metadata) pair. When the dataset shows up in the "Datasets" panel, it's ready to train against.

> ⏳ **On free-tier Colab T4 this can take a while.** Just leave it running until the dashboard says **Done!** — the encoding panel updates as each file finishes.


## 7. Train your first LoRA

In the dashboard tab, click **"+ Finetune"** (or "New finetune"). This kicks off a training subprocess that watches your dataset and produces a small adapter `.safetensors` file you can use in inference.

### Fill in the form

| Field | What to put | Why |
|---|---|---|
| **Name** | `my-first-lora` (or whatever — alphanumeric + hyphens) | Used as the run ID and the `.safetensors` filename |
| **Model** | `sa3-medium` (the one you downloaded in Step 3) | Base model to finetune against |
| **Dataset** | the one you created in Step 6 | Pre-encoded latents to train on |
| **LoRA type** | **`DoRA`** (recommended) | Generally better-quality fits than plain LoRA. `DoRA-XS` is the small-file variant. |
| **LoRA rank** | `16` | Capacity knob. 16 is a sane default. Higher = more parameters (heavier file, slower train, more overfitting risk). |
| **Steps** | `10000` | A reasonable LoRA lands around **10k steps** — that's where it *creatively underfits*: still general enough to vary on new prompts, not yet memorizing the training set. Past 10–20k it overfits and starts regurgitating. |
| **Batch size** | `1` on T4, up to `8` on H100 | Bigger = better gradients but more VRAM. T4 caps at 1 for SA3-medium fp16. |
| **Learning rate** | leave default | Usually fine; lower if you see grad-norm spikes |
| **Demo every** | `500` (or `1000`) | How often to generate audio previews during training. More frequent = more disk + more wall-clock spent on demos. |
| **Checkpoint every** | `500` | How often to save a `.safetensors` file. Each one is restartable. |

Click **Launch**.

### What you'll see during training

1. **Run appears** in the run panel with status `loading...` (5–15 s while the base model loads into VRAM)
2. Status flips to `training` and a **loss curve** starts plotting
3. Every `demo_every` steps, the run pauses for ~30 s to generate **4 demo MP3s** with their **spectrogram previews** (your monitor cell in Step 5 will tick `1 training` → keep an eye on it)
4. Every `checkpoint_every` steps, a fresh `.safetensors` lands and shows up in the checkpoints list

The dashboard tab is the live view. The status monitor cell in this notebook is the at-a-glance summary.

### How to know when to stop

Two signals — use both:

**1. Loss curve.** Watch the loss panel in the dashboard. The **elbow** — where it stops dropping steeply and starts to flatten — tends to be a good checkpoint to keep. Past the elbow you're refining, but also creeping toward memorization.

**2. Your ears, on the demos.** Open a demo MP3 every couple of thousand steps. What you're looking for in a healthy run:

- **Base RF demos (CFG≈7) light up first.** Around the time the run is starting to "get it," your CFG=7 demos suddenly sound right — clearly your training style on a coherent prompt.
- **Then CFG=7 over-cooks, and CFG=1 takes over.** Past the elbow, CFG=7 starts sounding artifacted / over-saturated. The lower-CFG demos (CFG≈1) keep improving and end up sounding the cleanest. If CFG=1 is sounding good and CFG=7 isn't, that's a sign the LoRA has internalized the style and no longer needs the prompt-classifier guidance to bring it out.
- **Conditional → unconditional crossover.** Early on, only the prompted demos sound like the training style. Later, even the **empty-prompt / unconditional** demos start sounding like the style — the model has "absorbed" the dataset, not just learned to recall it on cue.
- **ARC demos lag a bit but end up cleaner.** The ARC-distilled demos take a few thousand more steps to catch up to the base RF demos, but their final quality is usually better. Don't stop based on ARC alone too early; trust the base RF signal for "is this learning yet."

**3. Don't fear a memorized checkpoint — it might be exactly what you want.** "Overfitting" is only a problem if you're chasing creative variation. A memorized checkpoint can be incredibly useful:

- Dial **LoRA strength down** at inference time (in the gradio panel) — a memorized adapter at strength ~0.7 often gives the cleanest "in the style of" generations.
- **audio2audio / style-transfer** effects often hit harder with a memorized model — the strong style signal pulls the input audio into the training distribution more decisively.

So even past the elbow, save checkpoints. Different downstream uses want different points on the underfit ↔ memorize curve.

**Failure modes:**
- Demos sound **identical to the input** by 5k or so → overfitting too fast. Lower the rank or stop earlier.
- Demos sound **nothing like the input** even at 10k+ → dataset is too varied, too small, or LR is too low.

You can **stop the run anytime** (button in the dashboard) — the last checkpoint is yours. You can also **resume** from any checkpoint to train further.

### Get your LoRA out of Colab

1. In the dashboard's run panel, find the checkpoint you want
2. Click the download (⬇) button → grabs the `.safetensors` file (~20 MB for DoRA/LoRA at rank 16; DoRA-XS is smaller)
3. **Or** copy it from Drive:
   ```
   /content/drive/MyDrive/underfit-state/runs/<run-id>/<step>.safetensors
   ```

Now drop that `.safetensors` into any Stable Audio 3 inference setup (the dashboard's gradio panel, `python -m stable_audio_3`, ComfyUI's SA3 nodes, etc.) and the adapter layer-grafts onto SA3-medium at runtime.

### What to try next

- **DoRA-XS at rank 32** — tiny files, more capacity than rank-16 DoRA.
- **More steps** (15k, 20k) — push closer to memorization to see what creative underfitting looks like vs. overfitting.
- **Different prompts in demos** — edit the demo prompt list to test generalization.
- **A larger / more varied dataset** — usually the highest-leverage knob.
